In [1]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. CONFIGURATION & ASSET DNA
# ==========================================
MT5_PATH = r"C:\Program Files\SCFM MT5 Terminal\terminal64.exe"

SYMBOL_MAP = {
    'EURUSD=X': 'EURUSD.pro', 'GBPUSD=X': 'GBPUSD.pro', 'USDJPY=X': 'USDJPY.pro',
    'AUDUSD=X': 'AUDUSD.pro', 'NZDUSD=X': 'NZDUSD.pro', 'USDCAD=X': 'USDCAD.pro',
    'USDCHF=X': 'USDCHF.pro', 'EURGBP=X': 'EURGBP.pro',
    'GC=F':     'XAUUSD.pro',  'SI=F':     'XAGUSD.pro',  'USOIL': 'USOIL',
    'NAS100':   'NAS100.pro',  'SPX500':   'SP500.pro',   'US30':  'WS30.pro'
}

ASSET_DNA = {
    'EURUSD=X': {'pip': 0.0001, 'fee_pip': 2.0, 'atr_mult': 1.0, 'sl_buf_pip': 1.5, 'rsi_bull': 35, 'rsi_bear': 65},
    'GBPUSD=X': {'pip': 0.0001, 'fee_pip': 2.5, 'atr_mult': 1.0, 'sl_buf_pip': 1.5, 'rsi_bull': 35, 'rsi_bear': 65},
    'USDJPY=X': {'pip': 0.01,   'fee_pip': 2.5, 'atr_mult': 1.0, 'sl_buf_pip': 1.5, 'rsi_bull': 35, 'rsi_bear': 65},
    'AUDUSD=X': {'pip': 0.0001, 'fee_pip': 2.5, 'atr_mult': 1.0, 'sl_buf_pip': 1.5, 'rsi_bull': 35, 'rsi_bear': 65},
    'NZDUSD=X': {'pip': 0.0001, 'fee_pip': 3.0, 'atr_mult': 1.0, 'sl_buf_pip': 1.5, 'rsi_bull': 35, 'rsi_bear': 65},
    'USDCAD=X': {'pip': 0.0001, 'fee_pip': 2.5, 'atr_mult': 1.0, 'sl_buf_pip': 1.5, 'rsi_bull': 35, 'rsi_bear': 65},
    'USDCHF=X': {'pip': 0.0001, 'fee_pip': 2.5, 'atr_mult': 1.0, 'sl_buf_pip': 1.5, 'rsi_bull': 35, 'rsi_bear': 65},
    'EURGBP=X': {'pip': 0.0001, 'fee_pip': 3.0, 'atr_mult': 1.25,'sl_buf_pip': 1.5, 'rsi_bull': 35, 'rsi_bear': 65},
    'GC=F':     {'pip': 0.01,   'fee_pip': 3.5, 'atr_mult': 1.5, 'sl_buf_pip': 5.0, 'rsi_bull': 35, 'rsi_bear': 65},
    'SI=F':     {'pip': 0.001,  'fee_pip': 4.5, 'atr_mult': 1.75,'sl_buf_pip': 0.03,'rsi_bull': 20, 'rsi_bear': 80},
    'USOIL':    {'pip': 0.01,   'fee_pip': 3.0, 'atr_mult': 1.0, 'sl_buf_pip': 0.02,'rsi_bull': 35, 'rsi_bear': 65},
    'NAS100':   {'pip': 0.1,    'fee_pip': 2.5, 'atr_mult': 1.5, 'sl_buf_pip': 0.50,'rsi_bull': 35, 'rsi_bear': 65},
    'SPX500':   {'pip': 0.1,    'fee_pip': 2.5, 'atr_mult': 1.5, 'sl_buf_pip': 0.50,'rsi_bull': 35, 'rsi_bear': 65},
    'US30':     {'pip': 0.1,    'fee_pip': 2.5, 'atr_mult': 1.5, 'sl_buf_pip': 0.50,'rsi_bull': 35, 'rsi_bear': 65}
}

def fetch_mt5_h4(symbol, years=5, path=MT5_PATH):
    if not mt5.initialize(path=path): raise RuntimeError("MT5 Init Failed")
    if not mt5.symbol_select(symbol, True): return pd.DataFrame()
    rates = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_H4, 0, int(years * 6 * 250))
    if rates is None or len(rates) == 0: return pd.DataFrame()
    df = pd.DataFrame(rates)
    df["time"] = pd.to_datetime(df["time"], unit="s")
    return df.set_index("time")[["open", "high", "low", "close", "tick_volume"]].rename(
        columns={"open":"Open", "high":"High", "low":"Low", "close":"Close", "tick_volume":"Volume"})

class P3FunnelDebugger:
    def __init__(self, symbols, years=5):
        self.symbols = symbols
        self.years = years
        self.data_store = {}

    def fetch_and_prepare(self):
        print("▶ Fetching Data...")
        if not mt5.initialize(path=MT5_PATH): raise RuntimeError("MT5 Init Failed")
        for sym_raw in self.symbols:
            sym_mt5 = SYMBOL_MAP.get(sym_raw)
            if not sym_mt5: continue
            df = fetch_mt5_h4(sym_mt5, years=self.years)
            if df.empty: continue
            df.index = df.index.tz_localize('UTC').tz_convert('Europe/London')
            
            # Indicators
            tr = pd.concat([df['High']-df['Low'], abs(df['High']-df['Close'].shift(1)), abs(df['Low']-df['Close'].shift(1))], axis=1).max(axis=1)
            df['ATR'] = tr.ewm(alpha=1/14, adjust=False).mean()
            delta = df['Close'].diff()
            gain, loss = delta.clip(lower=0), -delta.clip(upper=0)
            with np.errstate(divide='ignore', invalid='ignore'):
                rs = gain.ewm(alpha=1/14, adjust=False).mean() / loss.ewm(alpha=1/14, adjust=False).mean()
                df['RSI'] = (100 - (100 / (1 + rs))).replace([np.inf, -np.inf], 100).fillna(50)
            df['EMA200'] = df['Close'].ewm(span=200, adjust=False).mean()
            
            self.data_store[sym_raw] = df

    def run_funnel_analysis(self):
        print("\n🔍 RUNNING FUNNEL ANALYSIS (FINDING THE BOTTLENECK)...")
        print("="*100)
        
        for sym_raw, df in self.data_store.items():
            dna = ASSET_DNA.get(sym_raw, ASSET_DNA['EURUSD=X'])
            o, c, h, l = df['Open'], df['Close'], df['High'], df['Low']
            body = (c - o).abs().replace(0, np.nan)
            u_wick, l_wick = h - np.maximum(o, c), np.minimum(o, c) - l
            candle_range = h - l

            # STEP 1: Pattern Geometry
            # Hammer: Wick >= 2x Body, Upper Wick <= 0.25x Body, Close in top 25%
            is_hammer = (l_wick >= 2 * body) & (u_wick <= 0.25 * body) & (c >= l + 0.75 * candle_range)
            # Engulf: Current Body > Prev Body, Engulfing logic
            prev_body = body.shift(1)
            is_bull_engulf = (c > o) & (c.shift(1) < o.shift(1)) & (c > o.shift(1)) & (o < c.shift(1)) & (body > prev_body)
            
            step1_pattern = (is_hammer | is_bull_engulf).sum()
            if step1_pattern == 0: continue

            # STEP 2: Trend + Volatility + Session
            vol_gate = candle_range > (df['ATR'] * dna['atr_mult'])
            trend_bull = c > df['EMA200']
            is_weekday = df.index.weekday < 5
            is_fri_pm = (df.index.weekday == 4) & (df.index.hour >= 16)
            sess_gate = is_weekday & ~is_fri_pm & (df.index.hour >= 8) & (df.index.hour < 21)
            
            step2_trend_vol = (is_hammer & trend_bull & vol_gate & sess_gate).sum()
            
            # STEP 3: ADX (Trend Strength)
            # Calculate ADX quickly for funnel
            up = df['High'].diff(); dn = -df['Low'].diff()
            pdm = np.where((up>dn)&(up>0), up, 0.0); mdm = np.where((dn>up)&(dn>0), dn, 0.0)
            a = 1/14
            pdi = 100*pd.Series(pdm, index=df.index).ewm(alpha=a, adjust=False).mean() / df['ATR']
            mdi = 100*pd.Series(mdm, index=df.index).ewm(alpha=a, adjust=False).mean() / df['ATR']
            dx = 100*np.abs(pdi-mdi)/(pdi+mdi+1e-10)
            adx = dx.ewm(alpha=a, adjust=False).mean()
            
            step3_adx = (is_hammer & trend_bull & vol_gate & sess_gate & (adx >= 20)).sum()

            # STEP 4: Momentum (RSI Filter)
            # This is usually the killer. RSI < 45 while Price > EMA200
            mom_hammer_bull = df['RSI'] < 45 
            step4_mom = (is_hammer & trend_bull & vol_gate & sess_gate & (adx >= 20) & mom_hammer_bull).sum()

            # STEP 5: Confirmation Logic (Close of next candle > High of pattern)
            # We simulate: if Signal is True at index i, did Close[i+1] > High[i]?
            # We shift the pattern mask by -1 to see if the *next* candle closed higher
            confirmed_signals = (step4_mom > 0) # Placeholder logic for counting
            
            # To count confirmation accurately:
            # We take the boolean mask of signals from Step 4, and check the NEXT candle's close
            mask_step4 = (is_hammer & trend_bull & vol_gate & sess_gate & (adx >= 20) & mom_hammer_bull).fillna(False)
            # Check if Close at i+1 > High at i
            next_close = df['Close'].shift(-1)
            pattern_high = df['High']
            confirmed_count = (mask_step4 & (next_close > pattern_high)).sum()

            print(f"📊 {sym_raw:12} | Step 1 (Pattern): {step1_pattern:4} | "
                  f"Step 2 (Trend/Vol/Sess): {step2_trend_vol:4} | "
                  f"Step 3 (+ADX): {step3_adx:4} | "
                  f"Step 4 (+RSI<45): {step4_mom:4} | "
                  f"Step 5 (+Confirm): {confirmed_count:4}")
            
            if step4_mom > 0 and confirmed_count == 0:
                print(f"   ⚠️  BOTTLENECK DETECTED: Signals found, but Confirmation Failed. "
                       f"Price is not breaking the pattern high in the next candle.")
            elif step4_mom == 0 and step3_adx > 0:
                print(f"   ⚠️  BOTTLENECK DETECTED: ADX passed, but RSI Filter killed all signals.")

# ==========================================
# EXECUTION
# ==========================================
if __name__ == "__main__":
    debugger = P3FunnelDebugger(symbols=['EURUSD=X', 'GC=F', 'NAS100'], years=5)
    debugger.fetch_and_prepare()
    debugger.run_funnel_analysis()
    mt5.shutdown()

▶ Fetching Data...

🔍 RUNNING FUNNEL ANALYSIS (FINDING THE BOTTLENECK)...
📊 EURUSD=X     | Step 1 (Pattern):  350 | Step 2 (Trend/Vol/Sess):   11 | Step 3 (+ADX):    8 | Step 4 (+RSI<45):    0 | Step 5 (+Confirm):    0
   ⚠️  BOTTLENECK DETECTED: ADX passed, but RSI Filter killed all signals.
📊 GC=F         | Step 1 (Pattern):  406 | Step 2 (Trend/Vol/Sess):    3 | Step 3 (+ADX):    2 | Step 4 (+RSI<45):    0 | Step 5 (+Confirm):    0
   ⚠️  BOTTLENECK DETECTED: ADX passed, but RSI Filter killed all signals.
📊 NAS100       | Step 1 (Pattern):  497 | Step 2 (Trend/Vol/Sess):   14 | Step 3 (+ADX):   10 | Step 4 (+RSI<45):    5 | Step 5 (+Confirm):    2
